# Practical Python with Semantic Python and OpenAI

A support-operations workflow covering values, collections, operators, control flow, functions, classes, exceptions, context managers, and asynchronous execution. Semantic comparisons use the OpenAI backend; repeated identical comparisons use the decision cache.

## 1. OpenAI credentials

The key is read from `OPENAI_API_KEY` or requested with `getpass`.

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

## 2. Values, collections, variables, and operators

In [ ]:
ticket = {
    "id": 1042,
    "customer": "Northwind",
    "priority": 2,
    "active": True,
    "owner": None,
    "tags": ["account", "workflow"],
}
service_levels = (4, 8, 24)
supported_tags = {"account", "billing", "workflow"}

ticket_id, priority = ticket["id"], ticket["priority"]
is_priority = ticket["active"] and priority <= 2
has_supported_tag = bool(set(ticket["tags"]) & supported_tags)
response_window = service_levels[priority - 1]

print(f"Ticket {ticket_id}: {response_window}h response window")
print("Priority workflow:", is_priority and has_supported_tag)

## 3. Backend configuration

Probabilities between `0.15` and `0.85` raise `UncertainDecisionError`.

In [ ]:
from semantic_python import (
    OpenAIBackend,
    Semantic,
    UncertainDecisionError,
    configure,
)

MODEL = "gpt-6-luna"
STOP_REQUEST = "the user wants to stop"

configure(
    backend=OpenAIBackend(model=MODEL),
    true_threshold=0.85,
    false_threshold=0.15,
    uncertainty="raise",
    cache=True,
)

## 4. Semantic classification

Each result retains its probability, backend, model, provenance, and cache status.

In [ ]:
stop_text = "Please save the current work and close this session."
continue_text = "Continue processing the remaining support tickets."

stop_message = Semantic(stop_text)
continue_message = Semantic(continue_text)
stop_decision = stop_message == STOP_REQUEST
continue_decision = continue_message == STOP_REQUEST

for label, decision in [("stop", stop_decision), ("continue", continue_decision)]:
    print(
        label,
        f"value={decision.value}",
        f"probability={decision.probability:.2f}",
        f"model={decision.model}",
        f"cached={decision.cached}",
    )

## 5. Control flow

In [ ]:
try:
    if stop_message == STOP_REQUEST:
        action = "close-session"
    elif continue_message == STOP_REQUEST:
        action = "pause-queue"
    else:
        action = "continue-queue"
except UncertainDecisionError:
    action = "human-review"

print("Selected action:", action)

messages = [continue_message, stop_message]
for message in messages:
    if message == STOP_REQUEST:
        print("Loop stopped on:", message.value)
        break

pending = iter(messages)
current = next(pending)
while current != STOP_REQUEST:
    print("Processed:", current.value)
    current = next(pending)

## 6. Functions, comprehensions, and generators

In [ ]:
def decide_stop(text: str, proposition: str = STOP_REQUEST):
    return Semantic(text) == proposition


def decision_values(items):
    for item in items:
        yield (item == STOP_REQUEST).value


function_result = decide_stop(stop_text)
decision_map = {message.value: message == STOP_REQUEST for message in messages}

print("Function result:", function_result.value, function_result.cached)
print("Comprehension:", {text: item.value for text, item in decision_map.items()})
print("Generator:", list(decision_values(messages)))

## 7. Classes, context management, and exceptions

In [ ]:
from contextlib import contextmanager
from dataclasses import dataclass


@dataclass(slots=True)
class StopClassifier:
    proposition: str = STOP_REQUEST

    def classify(self, text: str):
        return Semantic(text) == self.proposition


@contextmanager
def audit_section(name: str):
    print(f"Begin {name}")
    try:
        yield
    finally:
        print(f"End {name}")


classifier = StopClassifier()
with audit_section("classification"):
    try:
        class_result = classifier.classify(stop_text)
        print(class_result.value, class_result.probability, class_result.cached)
    except UncertainDecisionError as error:
        print("Review required:", error.decision.probability)

## 8. Asynchronous execution and operator semantics

The backend is synchronous; `asyncio.to_thread` moves the comparison off the event loop.

In [ ]:
import asyncio


async def check_without_blocking(message):
    return await asyncio.to_thread(lambda: message == STOP_REQUEST)


async_result = await check_without_blocking(stop_message)
negated = stop_message != STOP_REQUEST

print("Async decision:", async_result.value, async_result.cached)
print("Negated decision:", negated.value, negated.probability)
print("Identity:", stop_message is stop_message)

try:
    hash(stop_message)
except TypeError as error:
    print("Semantic values are intentionally unhashable:", error)